# <font color='#569098' size=20> ESMC and ESMFold2 on the Cluster
 </font>

Tutorial by Nathaniel Felbinger

Additional Reading:
* [ESMC/ESMFold2 Preprint](https://www.biorxiv.org/content/10.64898/2026.06.03.729735v1),
* [ESM3 Paper](https://www.science.org/doi/10.1126/science.ads0018),
* [ESM Atlas](https://biohub.ai/esm/protein/atlas)




---

### **OBJECTIVES**

By the end of this notebook you should be able to:

* <font color='#e59454'>Extract ESM logits as a mean to score sequences and mutants</font>
* <font color='#e59454'>Perform structure predictions with ESMFold2</font>

### **TUTORIAL**
The tutorial will have you perform the following tasks:
* <font color='#e59454'>You will use the EMSC language model to predict favorability of point mutants in a simple monomer protein and compare predictions to experimental data</font>
* <font color='#e59454'>Make structure predictions with ESMFold2 and assess model outputs</font>
* <font color='#e59454'>Use built-in feature extraction tool to assess functional and structure features predicted by ESM</font>



---

**_Please save a copy of this notebook to your own Google Drive before beginning._**

# <font color='#569098'>What is ESM?</font>

ESM (Evolutionary Scale Model) is a family of protein language models (PLMs). These models are trained on protein sequences to learn higher-level representations of proteins including insights into structure, function, fitness, and evolutionary mappings. The current generation language model is ESMC (ESM Cambrian) along with the multi-modal ESM3 model. ESMFold2 is a structure-prediction tool built on top of the knowledge of ESMC. Current benchmarking indicates it is at least competitive with AlphaFold3.  

# <font color='#569098'>Setup</font>


### Activate the esmfold2 environment

Note that conda environments are case-sensitive
To check your installed environments you can run:
conda info --envs


```bash
conda activate esmfold2

#conda info --envs


# <font color='#569098'>Investigating an Unknown Protein</font>

> You have been given the protein sequence below and are tasked with finding out its identity. Unfortunately, every tool that could quickly identify the protein for you is currently not available due to mysterious circumstances. You will have to perform your own investigation to characterize this mysterious protein sequence.



> This is a real protein. Please do not spoil the outcome by searching databases like BLAST, uniprot, or RCSB. Treat it like you are investigating an uncharacterized protein or de novo sequence.


> The sequence:
**AVESIEKKRIRKGRVEYLVKWRGWSPKYNTWEPEENILDPRLLIAFQNRE**















##Preliminary Work Before ESM Analysis

In the typical in-silico protein characterization workflow of a structural biologist, one would start by searching databases for homologous sequences and existing structures. However, sometimes when working with wet-lab collaborators it can be nice to provide more "traditional" sequence-based predictions of protein qualities. While they are redundant (or often less accurate) than what can be gained from analyzing predicted structures, it may be more interpretable to an AI-apprehensive scientist.


Some of these informative protein qualities one can predict from sequence and map to individual residues include:

*   Flexibility
*   Polarity
*   Hydrophobicity
*   Alpha-helix/beta-sheet propensity predictions






**Use the [Expasy ProtScale web server](https://web.expasy.org/protscale/) to generate some sequence-based predictions about the qualities of the protein.** Some qualities you can focus on include:


*   Hydrophilicity (Hope&Woods): Broad stretches of residues with low hydrophilicity may indicate transmembrane/membrane-interacting residues. A quick drop off in hydrophilicity or a hydrophobic region flanked by hydrophilic regions may also indicate the
presence of well-folded region with stable secondary structure.

*   Average Flexibility: As the name suggests, higher values indicate more flexible residues. Flexible stretches of proteins may indicate disorganized coil regions.

*   Beta-sheet (Levitt): Propensity of residues to fold into beta-sheets. Higher values indicate more likely beta-sheet formation.

*   Alpha-helix (Chou Fasman): Propensities of residues to fold into alpha-helices. Higher values indicate more likely alpha-helix formation.

*   Bulkiness: Maps the residue side chain volumes occupying each position. More bulky regions indicate regions of charged residues and/or aromatic residues. Certain binding-motifs are found in such regions.






## <font color="#B03030">Preliminary Analysis Questions</font>


**Question 1.** Jot down some basic observations about sequence. Are there any stretches of charged or hydrophobic residues? Are there residues that have distinct contributions to protein structure and function (PRO, CYS, GLY)?

**Answer:**


**Question 2.** Can you make any predictions about structural qualities of the protein? Is there support for any secondary structure or is it disorganized?

**Answer:**



Answer 1 Here;


Answer 2 here:

##ESMC Feature Extraction

>We will now utilize ESMC to analyze the sequence and extract features using a complementary sparse autoencoder (SAE) model.

>When ESMC analyzes a protein sequence, each residue is encoded in a dense and complex vector of values. The features encoded in these vectors is hard to interpret, as different dimensions of these vector stacks can represent completely unrelated information. SAEs work by expanding this vector space in a way that such features are "further apart" from one another and the number of active features is capped. The sparse reconstructions of ESM's internal representations are then mapped to database entries.

>As ESMC embeds each residue of the sequence, these features are also found at a per-residue level. So one can extract features from individual residues (this is a charged residue) or aggregate continuous activation patterns for sets of residues to get motif-level (these 7 residues are part of a EF-hand motif) or broad functional/structural level (these 30 residues are in a proline-capped alpha-helix).

On your cluster, make a directory that you can place content of this activity into (e.g. bootcamp_esmfold2)

Copy the esmc_feature_analysis.py script from the ESM_activity folder into your directory and run it. By default the script will print the features with the top 5 strongest activations in 10 residue long windows. So the features you will observe are motif-level or smaller structural/functional-level features.

If you want to adjust the printed features, feel free to change the window size by adjusting the "MIN_CONSECUTIVE_RESIDUES" variable on line 19. You may also need to adjust the "ACTIVATION_THRESHOLD" variable if you stop getting any hits.

Run the esmc_feature_analysis.py program (which has our sequence loaded by default) and analyze the features. Answer the questions below.

## <font color="#B03030">Feature Analysis Questions</font>

**Please answer the questions below.**


**Question 1.** Running the script with default settings, what are some of the default features being output. Looking at the "exemplars" section, what classes of proteins or specific proteins do you observe in two or more features. Are there any particular key words you see repeated?

**Answer:**


**Question 2.** Adjust the length of the extraction windows. Do you notice any difference in the qualities that the features describe as you decrease and increase this window?

**Answer:**

**Question 3.** Do you have any thoughts about what the identity of the protein might be? Do you have any predictions about potential structural features?

**Answer:**







##1. Structure Prediction with ESMFold2

> It is impressive how much one can predict from sequence-only methods alone. Perhaps these models may not be as much as a black box as people say? In reality, the precision of features extracted from analyzing sequences in this manner has not been thoroughly benchmarked. Tools like ESMC are more often used as a foundation for fine-tuned models, with zero-shot prediction of sequences like this not commonly found in design/analysis pipelines (yet). We will take a more common approach to further analyze the protein by predicting its structure.

Copy the mystery_esmfold2.py script from the ESM_activity folder into your the directory you performed the previous step in. This script will produce three predictions for the mystery sequence and output basic confidence statistics.

Execute the mystery_esmfold2.py script and download the three output predictions. Open the predictions in pymol and answer the questions below.

## <font color="#B03030">Modeling Analysis Questions</font>

**Please answer the questions below.**


**Question 1.** In a few sentences, describe the composition of the predictions.  What sort of secondary structure is present? Do any of the structural characteristics align with predictions from the first two parts of this analysis?

**Answer:**


**Question 2.** Dropout is the process of obscuring certain information from the model (in this case hiding some of the language model embeddings). Certain approaches utilize drop out to improve heterogenity of the output. Just as a fun experiment, adjust the lm_dropout setting (line 39) from 0.1 to 0.7 and run the script again. Do you notice any change in the confidence metrics? Any change in the outputs?

**Answer:**

**Question 3.** Predict the structure using the AlphaFold3 web server. Use pymol to compare the highest-confidence ESM3 prediction with the AlphaFold3 prediction. How do they compare?

**Answer:**

##Solving the Mystery

>Access to BLAST has been restored! However, before we blast our sequence to learn its identity, let's explore a couple alternative databases to see if we can get the answer through other means.

>The ESM Atlas is a database of proteins characterized using the ESM language models. The databases are massive, with a mix of in-silico entries and entries with associated experimental data. This can serve as a fun way to gain potential insights into a protein sequence without well-characterized homologous sequences.

>Foldseek is a tool that enables comparison of 3d structure. This can be used to cluster structures together and search for similar folds in a database of structures/models. We (hopefully) touch on this in some more depth at some point. But for now, just know that there is a webserver that enables you to search for structures with a similar fold to an uploaded .cif/.pdb file.



## <font color="#B03030">Answers to Our Mystery</font>

**Please answer the questions below.**

**Question 1.** Search the sequence in the ESM Atlas server  ([LINK](https://biohub.ai/esm/protein/atlas)). You can just input the sequence, but I would prompt it by specifically asking for similar clusters to the sequence.

**Answer:** What sort of proteins did the ESM Atlas report are similar to the input sequence?


**Question 2.** Upload the highest-confidence ESMFold2 prediction the foldseek search webserver ([LINK](https://search.foldseek.com/search)) and hit the "SEARCH" button. You should see a list of databases in a row above "Top Hits". Scroll all the way to the right and click "PDB100". This reports hits against experimentally-determined structures from the RCSB database. What structures appear to align with the model of our mystery sequence?

**Answer:**

**Question 3.** Blast our mystery sequence against the PDB database ([LINK](https://blast.ncbi.nlm.nih.gov/Blast.cgi)). What is the top hit? What is our mystery protein?

**Answer:**
